In [ ]:
## 初期import

from pathlib import Path

import numpy as np 
import pandas as pd
import japanize_matplotlib

# notebooks/ からの相対パス（data/raw, data/external, data/processed）
PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

train = pd.read_csv(RAW_DIR / "train.csv", encoding="utf-8-sig")
test = pd.read_csv(RAW_DIR / "test.csv", encoding="utf-8-sig")
sample_submission = pd.read_csv(RAW_DIR / "sample_submission.csv", encoding="utf-8-sig")

train["PitNextLap"] = train["PitNextLap"].astype(int)

Using Python 3.12.13 environment at: C:\Users\la9ma\AnotherBrainContainer\10_kaggle\06_predicting_f1_pit__stops\.venv
Resolved 12 packages in 3.95s
   Building japanize-matplotlib==1.1.3
      Built japanize-matplotlib==1.1.3
Prepared 1 package in 875ms
Installed 1 package in 24ms
 + japanize-matplotlib==1.1.3


In [ ]:
# ## SWEETVIZ を用いた EDA (1) : Train vs Test

# !pip install -q -U sweetviz
# import sweetviz as sv
# from sweetviz import FeatureConfig

# report = sv.compare(
#     [train,"Train"],
#     [test,"Test"],
#     target_feat = "PitNextLap"
# )

# report.show_html(str(EXTERNAL_DIR / "sv_train_vs_test.html"))


# ## SWEETVIZ を用いた EDA (2) : 0 vs 1

# !pip install -q -U sweetviz
# import sweetviz as sv
# from sweetviz import FeatureConfig

# train_0 = train[train["PitNextLap"] == 0]
# train_1 = train[train["PitNextLap"] == 1]

# report = sv.compare(
#     [train_0,"Train_0"],
#     [train_1,"Train_1"],
#     target_feat = "PitNextLap"
# )

# report.show_html(str(EXTERNAL_DIR / "sv_0_vs_1.html"))


## 可視化

In [ ]:
## カテゴリ変数と目的変数の割合（Race / Compound のみ。Driver はカーディナリティが高いため除外）

import seaborn as sns
import matplotlib.pyplot as plt

target = "PitNextLap"
cat_for_heatmap = ["Race", "Compound"]  # Driver は除外
num = train.select_dtypes(include=["int64", "float64"]).columns.tolist()

# 学習データ全体でのクラス割合（リフトの分母になる）
# ※ 行ごとの「濃さ」は変わらず、列 k ごとに「全体の P(k) と同じか」を見るための基準。
p1_global = train[target].mean()
p0_global = 1.0 - p1_global

# Lift（倍率）の定義（列 k = 0 または 1）:
#   lift(k | 水準) = P(target=k | その水準) / P(target=k 全体)
# - crosstab(..., normalize="index") で得た値が分子 P(target=k | 水準)
# - 分母はデータ全体の P(k)。割り算で「全体比何倍か」になる。
# - lift = 1.0 → その水準は全体と同じ寄り、>1 → そのクラスが濃い、<1 → 薄い。
fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(8, max(10, train["Race"].nunique() * 0.28))) #経験則での0.28インチ比確保

for ax, col in zip(axes, cat_for_heatmap):
    # ① 行内条件付き確率（各行で 0 と 1 の割合の合計は 1）
    pct = pd.crosstab(train[col], train[target], normalize="index") #indexで行内の割合を出す
    pct = pct.reindex(columns=[0, 1], fill_value=0) #0,1のみのデータにする
    
    # ② 列ごとに全体割合で割って lift 行列にする（0 列は P(0) で、1 列は P(1) で割る）
    lift = pd.DataFrame(
        {0: pct[0] / p0_global, 1: pct[1] / p1_global},
        index=pct.index,
    )
    sns.heatmap(
        lift,
        annot=True,
        fmt=".2f",
        cmap="RdYlBu_r", #liftと相性がいい色
        center=1.0,
        cbar_kws={"label": "lift = P(k|水準) / P(k)"},
        ax=ax,
    )
    ax.set_title(
        f"{col} × {target}（lift）  全体: P(0)={p0_global:.3f}, P(1)={p1_global:.3f}"
    )
    ax.set_ylabel(col)
    ax.set_xlabel(f"{target} のクラス k（各セルは P(k|水準)÷P(k)）")

plt.tight_layout()
plt.show()

## 連続変数同士の相関行列の可視化

Nr = train[num].drop(columns=["id"], errors="ignore")

fig2, ax2 = plt.subplots(figsize=(12, 10))
sns.heatmap(Nr.corr(), annot=True, cmap="coolwarm", ax=ax2)
ax2.set_title("連続変数同士の相関行列")
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 【結論】カテゴリ水準ごとの「1 の割合 P(1|水準)」を横棒（Plotly）で見る表現の優位性
# =============================================================================
# 1. 解釈が直線的: 横軸が確率なので、水準間の大小・全体からの距離が数値として読み取りやすい。
#    （ヒートマップは色の段階比較になりがちで、細かい差の順位付けは棒の方が向くことが多い。）
# 2. 全体基準の重ね合わせ: 全体の P(1) を縦の参照線にすると「全体より濃い／薄い」が一発で分かる。
# 3. インタラクション: Plotly はホバーで n（件数）を出せるため、件数が少ない水準の偶然の高さに
#    騙されにくい。（リフトや割合の可視化では母数の提示が特に重要。）
# 4. 探索性: ズーム・パン・凡例クリックで水準が多い Race でも局所を掘れる。
#
# 【注意】棒の長さは「その水準の中での 1 の割合」であり、リフト（全体比）ではない。
#         全体比は「参照線との差」で読むか、ホバーで n と併せて判断する。
# =============================================================================

import plotly.graph_objects as go
from plotly.subplots import make_subplots

target = "PitNextLap"
features = ["Race", "Compound"]  # Driver はカーディナリティが高いため除外（必要なら列を追加）
p1_global = float(train[target].mean())

fig = make_subplots(
    rows=len(features),
    cols=1,
    subplot_titles=[
        f"{col}: P({target}=1 | {col})　（破線＝全体の P(1)={p1_global:.4f}）"
        for col in features
    ],
    vertical_spacing=0.08,
)

row_heights = []
for i, col in enumerate(features, start=1):
    g = train.groupby(col, dropna=False)[target].agg(["mean", "count"]).reset_index()
    g.columns = [col, "p1", "n"]
    g = g.sort_values("p1", ascending=True)
    row_heights.append(max(320, 14 * len(g)))

    fig.add_trace(
        go.Bar(
            y=g[col].astype(str),
            x=g["p1"],
            orientation="h",
            text=[f"{v:.3f} (n={int(n)})" for v, n in zip(g["p1"], g["n"])],
            textposition="outside",
            cliponaxis=False,
            hovertemplate=(
                f"<b>{col}=%{{y}}</b><br>"
                "P(1)=%{x:.4f}<br>"
                "n=%{customdata}<extra></extra>"
            ),
            customdata=g["n"],
            showlegend=False,
        ),
        row=i,
        col=1,
    )
    fig.add_vline(
        x=p1_global,
        line_width=2,
        line_dash="dash",
        line_color="rgba(80,80,80,0.9)",
        row=i,
        col=1,
    )

fig.update_xaxes(title_text="P(1 | 水準)", range=[0, 1], tickformat=".0%")
fig.update_layout(
    title="カテゴリ水準ごとの「1」になる割合（横棒）",
    height=int(sum(row_heights)),
    margin=dict(l=120, r=80, t=80, b=40),
    paper_bgcolor="white",
    plot_bgcolor="white",
)
fig.show()

## 前処理

In [5]:
## 前処理（外れ値のクリッピング）

# 分位数は **train のみ** で算出し、同じ閾値を train / test に適用（リーク防止）。
# クリップ後は **元列をドロップ**し、`_clipped` と「クリップ対象だった」**0/1 フラグ**を残す。

import numpy as np

COL_LAP = "LapTime (s)"
COL_DELTA = "LapTime_Delta"
COL_CUMDEG = "Cumulative_Degradation"

_nan_train = {c: int(train[c].isna().sum()) for c in [COL_LAP, COL_DELTA, COL_CUMDEG]}
_nan_test = {c: int(test[c].isna().sum()) for c in [COL_LAP, COL_DELTA, COL_CUMDEG]}
print("欠損数 train:", _nan_train)
print("欠損数 test:", _nan_test)

q95_lap = float(train[COL_LAP].quantile(0.95))
q5_delta = float(train[COL_DELTA].quantile(0.05))
q95_delta = float(train[COL_DELTA].quantile(0.95))
q95_cumdeg = float(train[COL_CUMDEG].quantile(0.95))

print(
    "train 由来の閾値:",
    {
        "LapTime q95": q95_lap,
        "LapTime_Delta q05": q5_delta,
        "LapTime_Delta q95": q95_delta,
        "Cumulative_Degradation q95": q95_cumdeg,
    },
)

COL_LAP_CLIP = f"{COL_LAP}_clipped"
COL_DELTA_CLIP = f"{COL_DELTA}_clipped"
COL_CUM_CLIP = f"{COL_CUMDEG}_clipped"
FLAG_LAP = f"{COL_LAP}_clip_flag"
FLAG_DELTA = f"{COL_DELTA}_clip_flag"
FLAG_CUM = f"{COL_CUMDEG}_clip_flag"

train[COL_LAP_CLIP] = np.minimum(train[COL_LAP], q95_lap)
test[COL_LAP_CLIP] = np.minimum(test[COL_LAP], q95_lap)

train[COL_DELTA_CLIP] = train[COL_DELTA].clip(lower=q5_delta, upper=q95_delta)
test[COL_DELTA_CLIP] = test[COL_DELTA].clip(lower=q5_delta, upper=q95_delta)

train[COL_CUM_CLIP] = np.minimum(train[COL_CUMDEG], q95_cumdeg)
test[COL_CUM_CLIP] = np.minimum(test[COL_CUMDEG], q95_cumdeg)

train[FLAG_LAP] = (train[COL_LAP] > q95_lap).astype("int8")
test[FLAG_LAP] = (test[COL_LAP] > q95_lap).astype("int8")

train[FLAG_DELTA] = (
    (train[COL_DELTA] < q5_delta) | (train[COL_DELTA] > q95_delta)
).astype("int8")
test[FLAG_DELTA] = (
    (test[COL_DELTA] < q5_delta) | (test[COL_DELTA] > q95_delta)
).astype("int8")

train[FLAG_CUM] = (train[COL_CUMDEG] > q95_cumdeg).astype("int8")
test[FLAG_CUM] = (test[COL_CUMDEG] > q95_cumdeg).astype("int8")

train.drop(columns=[COL_LAP, COL_DELTA, COL_CUMDEG], inplace=True)
test.drop(columns=[COL_LAP, COL_DELTA, COL_CUMDEG], inplace=True)

print(
    "保持列（クリップ後・フラグ）:",
    COL_LAP_CLIP,
    COL_DELTA_CLIP,
    COL_CUM_CLIP,
    FLAG_LAP,
    FLAG_DELTA,
    FLAG_CUM,
    sep="\n  ",
)
print("clip_flag 内訳（1=元値が閾値外でクリップ対象）:")
for name in (FLAG_LAP, FLAG_DELTA, FLAG_CUM):
    print(name, train[name].value_counts().sort_index().to_dict())

欠損数 train: {'LapTime (s)': 0, 'LapTime_Delta': 0, 'Cumulative_Degradation': 0}
欠損数 test: {'LapTime (s)': 0, 'LapTime_Delta': 0, 'Cumulative_Degradation': 0}
train 由来の閾値: {'LapTime q95': 109.72604999999999, 'LapTime_Delta q05': -24.819000000000003, 'LapTime_Delta q95': 19.026, 'Cumulative_Degradation q95': 84.40100000000001}
保持列（クリップ後・フラグ）:
  LapTime (s)_clipped
  LapTime_Delta_clipped
  Cumulative_Degradation_clipped
  LapTime (s)_clip_flag
  LapTime_Delta_clip_flag
  Cumulative_Degradation_clip_flag
clip_flag 内訳（1=元値が閾値外でクリップ対象）:
LapTime (s)_clip_flag {0: 417183, 1: 21957}
LapTime_Delta_clip_flag {0: 395234, 1: 43906}
Cumulative_Degradation_clip_flag {0: 417183, 1: 21957}


In [ ]:
# クリップ後の分布と clip_flag（Plotly・2列×3行）

import plotly.graph_objects as go
from plotly.subplots import make_subplots

COL_LAP_CLIP = "LapTime (s)_clipped"
COL_DELTA_CLIP = "LapTime_Delta_clipped"
COL_CUM_CLIP = "Cumulative_Degradation_clipped"
FLAG_LAP = "LapTime (s)_clip_flag"
FLAG_DELTA = "LapTime_Delta_clip_flag"
FLAG_CUM = "Cumulative_Degradation_clip_flag"

pairs = [
    (COL_LAP_CLIP, FLAG_LAP, "LapTime (s)"),
    (COL_DELTA_CLIP, FLAG_DELTA, "LapTime_Delta"),
    (COL_CUM_CLIP, FLAG_CUM, "Cumulative_Degradation"),
]

subplot_titles = []
for label in ["LapTime (s)", "LapTime_Delta", "Cumulative_Degradation"]:
    subplot_titles.append(f"{label}（クリップ後）")
    subplot_titles.append(f"{label} clip_flag")

fig = make_subplots(
    rows=3,
    cols=2,
    column_widths=[0.62, 0.38],
    subplot_titles=tuple(subplot_titles),
    vertical_spacing=0.07,
    horizontal_spacing=0.06,
)

for row, (clip_col, flag_col, label) in enumerate(pairs, start=1):
    fig.add_trace(
        go.Histogram(
            x=train[clip_col],
            nbinsx=120,
            showlegend=False,
            name=f"{label}_hist",
            marker_color="#3366cc",
        ),
        row=row,
        col=1,
    )
    vc = train[flag_col].value_counts().sort_index()
    xs = [str(int(i)) for i in vc.index]
    fig.add_trace(
        go.Bar(
            x=xs,
            y=vc.values,
            showlegend=False,
            name=f"{label}_flag",
            marker_color="#cc6633",
            textposition="outside",
            texttemplate="%{y:,}",
        ),
        row=row,
        col=2,
    )
    fig.update_xaxes(title_text="値", row=row, col=1)
    fig.update_yaxes(title_text="度数", row=row, col=1)
    fig.update_xaxes(title_text="0 / 1", row=row, col=2, type="category")
    fig.update_yaxes(title_text="行数", row=row, col=2)

fig.update_layout(
    height=980,
    title_text="クリップ後の分布と clip_flag（train）",
    paper_bgcolor="white",
    plot_bgcolor="white",
    bargap=0.25,
)
fig.show()

In [ ]:
## clip_flag × 目的変数 PitNextLap（クロス集計ヒートマップ）

# 各 clip_flag（0=閾値内のみ、1=クリップ対象だった）の水準ごとに、
# PitNextLap=0/1 の行内割合を見る。前処理セル実行済みが前提。

import matplotlib.pyplot as plt
import seaborn as sns

target = "PitNextLap"
flag_cols = [
    "LapTime (s)_clip_flag",
    "LapTime_Delta_clip_flag",
    "Cumulative_Degradation_clip_flag",
]

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(14, 4))

for ax, fcol in zip(axes, flag_cols):
    ct = pd.crosstab(train[fcol], train[target], normalize="index")
    ct = ct.reindex(columns=[0, 1], fill_value=0)
    sns.heatmap(
        ct,
        annot=True,
        fmt=".3f",
        cmap="RdYlBu_r",
        vmin=0,
        vmax=1,
        ax=ax,
        cbar_kws={"label": "P(target|flag)"},
    )
    ax.set_title(f"{fcol}
× {target}")
    ax.set_xlabel(target)
    ax.set_ylabel(fcol)

plt.suptitle("clip_flag × PitNextLap（train・normalize='index'）", y=1.05, fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
## ドライバー×レース 時系列グルーピング & スティント指標分析

import plotly.graph_objects as go
from plotly.subplots import make_subplots

LAP_COL = "LapTime (s)_clipped"

# ── Step 1: Driver カーディナリティ確認 ────────────────────────────────────
driver_counts = train["Driver"].value_counts()
n_unique = len(driver_counts)
is_one_to_one = (driver_counts == 1).all()

print(f"ユニークドライバー数: {n_unique}")
print(f"全行数: {len(train)}")
print(f"全ドライバーが1対1（各1行のみ）: {is_one_to_one}")
print("\n登場回数上位10ドライバー:")
print(driver_counts.head(10).to_string())

if is_one_to_one:
    print("\n→ 全ドライバーがデータに対して1対1 → 処理終了")
else:
    # ── Step 2: 複数回登場するドライバーを抽出（多い順） ──────────────────
    multi_drivers = driver_counts[driver_counts > 1].sort_values(ascending=False)
    print(f"\n複数ラップ登場するドライバー: {len(multi_drivers)} 名")

    # ── Step 3: (Driver, Race) でグルーピング → LapNumber でソート ─────────
    grp = (
        train[train["Driver"].isin(multi_drivers.index)]
        .sort_values(["Driver", "Race", "LapNumber"])
        .copy()
    )

    group_sizes = grp.groupby(["Driver", "Race"]).size()
    print(f"\n(Driver, Race) グループ数: {len(group_sizes)}")
    print("グループサイズ統計:")
    print(group_sizes.describe().to_string())

    # ── Step 4: スティント指標の算出 ──────────────────────────────────────
    # (Driver, Race, Stint) ごとに stint_length / lap_time_mean / lap_time_std を算出
    stint_stats = (
        grp.groupby(["Driver", "Race", "Stint"], sort=False)
        .agg(
            stint_length=("LapNumber", "count"),
            lap_time_mean=(LAP_COL, "mean"),
            lap_time_std=(LAP_COL, "std"),
            race_progress_start=("RaceProgress", "min"),
            race_progress_end=("RaceProgress", "max"),
        )
        .reset_index()
    )
    # スティント1ラップのみのケースは std=NaN → 0 で補完
    stint_stats["lap_time_std"] = stint_stats["lap_time_std"].fillna(0.0)

    print(f"\nスティント数（合計）: {len(stint_stats)}")
    print(stint_stats.head(10).to_string(index=False))

    # ── Step 5-A: 3指標の分布（ヒストグラム） ────────────────────────────
    metrics = [
        ("stint_length",  "スティントの長さ（ラップ数）", "ラップ", "#3366cc"),
        ("lap_time_mean", "平均ラップタイム（秒）",        "秒",     "#33aa66"),
        ("lap_time_std",  "ラップタイム標準偏差（秒）",    "秒",     "#cc6633"),
    ]

    fig1 = make_subplots(
        rows=1, cols=3,
        subplot_titles=[m[1] for m in metrics],
        horizontal_spacing=0.08,
    )
    for col_idx, (col_name, title, unit, color) in enumerate(metrics, start=1):
        fig1.add_trace(
            go.Histogram(
                x=stint_stats[col_name],
                nbinsx=60,
                marker_color=color,
                showlegend=False,
                hovertemplate=f"{unit}: %{{x}}<br>度数: %{{y}}<extra></extra>",
            ),
            row=1, col=col_idx,
        )
        fig1.update_xaxes(title_text=unit, row=1, col=col_idx)
        fig1.update_yaxes(title_text="度数", row=1, col=col_idx)

    fig1.update_layout(
        height=420,
        title_text="スティント指標の分布（Driver×Raceグルーピング）",
        paper_bgcolor="white",
        plot_bgcolor="white",
    )
    fig1.show()

    # ── Step 5-B: スティント番号ごとの指標推移（箱ひげ図） ──────────────
    fig2 = make_subplots(
        rows=1, cols=2,
        subplot_titles=[
            "スティント番号 × 平均ラップタイム",
            "スティント番号 × ラップタイム標準偏差",
        ],
        horizontal_spacing=0.08,
    )
    stint_nums = sorted(stint_stats["Stint"].unique())
    for col_idx, (y_col, y_label) in enumerate([
        ("lap_time_mean", "平均ラップタイム（秒）"),
        ("lap_time_std",  "ラップタイム標準偏差（秒）"),
    ], start=1):
        for s in stint_nums:
            vals = stint_stats.loc[stint_stats["Stint"] == s, y_col]
            fig2.add_trace(
                go.Box(
                    y=vals,
                    name=f"S{int(s)}",
                    boxmean=True,
                    showlegend=(col_idx == 1),
                ),
                row=1, col=col_idx,
            )
        fig2.update_yaxes(title_text=y_label, row=1, col=col_idx)
        fig2.update_xaxes(title_text="スティント番号", row=1, col=col_idx)

    fig2.update_layout(
        height=450,
        title_text="スティント番号ごとの指標推移",
        paper_bgcolor="white",
        plot_bgcolor="white",
    )
    fig2.show()

In [ ]:
## 特徴量エンジニアリング: 累積ラップタイム & 履歴スティント長 → Train / Test マージ

LAP_COL = "LapTime (s)_clipped"

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Part A: スティント内累積特徴量（expanding）― リーク回避
#   現ラップまでの情報だけを使うため未来情報を含まない
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def add_expanding_stint_features(df, lap_col=LAP_COL):
    sorted_df = df.sort_values(["Driver", "Race", "Stint", "LapNumber"])
    g = sorted_df.groupby(["Driver", "Race", "Stint"])[lap_col]
    sorted_df = sorted_df.copy()
    sorted_df["lap_time_cumean"]      = g.expanding().mean().reset_index(level=[0, 1, 2], drop=True)
    sorted_df["lap_time_custd"]       = g.expanding().std().fillna(0.0).reset_index(level=[0, 1, 2], drop=True)
    sorted_df["laps_in_stint_so_far"] = g.expanding().count().reset_index(level=[0, 1, 2], drop=True)
    return sorted_df.loc[df.index]  # 元の行順を復元

train = add_expanding_stint_features(train)
test  = add_expanding_stint_features(test)

print("【Part A】累積特徴量を追加")
check_cols = ["Driver", "Race", "Stint", "LapNumber",
              "lap_time_cumean", "lap_time_custd", "laps_in_stint_so_far"]
print(train[check_cols].head(8).to_string(index=False))

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Part B: 履歴スティント長（train のみで算出 → test にも適用）
#   test 側に train の正解情報を混ぜないためテーブルは train 由来のみ
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ① train の (Driver, Race, Stint) 単位でスティント長を集計
_stint_hist = (
    train.groupby(["Driver", "Race", "Stint"], sort=True)
    .agg(stint_length=("LapNumber", "count"))
    .reset_index()
    .sort_values(["Driver", "Race", "Stint"])
)

# ② 同一 (Driver, Race) 内で Stint 昇順に並べ、「現 Stint より前の全スティント」の累積平均を取る
#    expanding().mean() → Stint n までの平均 → shift(1) で現スティント自体を除外
_stint_hist["prev_stint_length_mean"] = (
    _stint_hist.groupby(["Driver", "Race"])["stint_length"]
    .expanding()
    .mean()
    .reset_index(level=[0, 1], drop=True)
    .shift(1)
)

# ③ Stint=1（前スティントなし）の NaN → train 全体の stint_length 中央値で補完
global_median_stint = float(_stint_hist["stint_length"].median())
_stint_hist["prev_stint_length_mean"] = _stint_hist["prev_stint_length_mean"].fillna(global_median_stint)

print(f"\n【Part B】履歴スティント長テーブル（先頭10行）")
print(_stint_hist.head(10).to_string(index=False))
print(f"  Stint=1 NaN 補完用グローバル中央値: {global_median_stint}")

# ④ Train / Test にマージ（key: Driver, Race, Stint）
merge_key = ["Driver", "Race", "Stint"]
train = train.merge(_stint_hist[merge_key + ["prev_stint_length_mean"]], on=merge_key, how="left")
test  = test.merge( _stint_hist[merge_key + ["prev_stint_length_mean"]], on=merge_key, how="left")

# ⑤ test に train 未登場の (Driver, Race, Stint) がある場合 → global_median で補完
train["prev_stint_length_mean"] = train["prev_stint_length_mean"].fillna(global_median_stint)
test["prev_stint_length_mean"]  = test["prev_stint_length_mean"].fillna(global_median_stint)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 確認
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
new_cols = ["lap_time_cumean", "lap_time_custd", "laps_in_stint_so_far", "prev_stint_length_mean"]
print(f"\n【NaN 確認】")
print("train:", train[new_cols].isna().sum().to_dict())
print("test: ", test[new_cols].isna().sum().to_dict())
print(f"\ntrain shape: {train.shape},  test shape: {test.shape}")

In [ ]:
## 特徴量重要度の算出 & test 予測（StratifiedKFold 4fold × LightGBM / 評価指標: AUC）

import lightgbm as lgb
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import plotly.graph_objects as go

TARGET    = "PitNextLap"
DROP_COLS = ["id", TARGET]
CAT_COLS  = ["Driver", "Compound", "Race"]

# ── エンコーダを train+test の全ラベルで fit ──────────────────────────────────
# test に train 未登場のドライバー・レースがあっても unknown 扱いを防ぐ
train_enc = train.copy()
test_enc  = test.copy()

for col in CAT_COLS:
    le = LabelEncoder()
    le.fit(pd.concat([train[col], test[col]]).astype(str))
    train_enc[col] = le.transform(train_enc[col].astype(str))
    test_enc[col]  = le.transform(test_enc[col].astype(str))

feature_cols = [c for c in train_enc.columns if c not in DROP_COLS]
X      = train_enc[feature_cols]
y      = train_enc[TARGET]
X_test = test_enc[feature_cols]

print(f"特徴量数: {len(feature_cols)}")
print("特徴量一覧:", feature_cols)

# ── LightGBM パラメータ（評価指標: AUC） ────────────────────────────────────
scale_pos_w = float((y == 0).sum() / (y == 1).sum())

lgb_params = {
    "objective":         "binary",
    "metric":            "auc",
    "learning_rate":     0.05,
    "num_leaves":        63,
    "min_child_samples": 20,
    "feature_fraction":  0.8,
    "bagging_fraction":  0.8,
    "bagging_freq":      1,
    "scale_pos_weight":  scale_pos_w,
    "verbose":           -1,
    "random_state":      42,
}
print(f"\nscale_pos_weight = {scale_pos_w:.3f}")

# ── StratifiedKFold 4fold 学習：重要度の収集と test 予測を同時に行う ──────────
N_SPLITS   = 4
skf        = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
importance_df = pd.DataFrame({"feature": feature_cols})
test_preds    = np.zeros(len(X_test))
oof_auc       = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=CAT_COLS)
    dval   = lgb.Dataset(X_val, label=y_val, categorical_feature=CAT_COLS, reference=dtrain)

    model = lgb.train(
        lgb_params,
        dtrain,
        num_boost_round=1000,
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=False),
        ],
    )

    auc = roc_auc_score(y_val, model.predict(X_val))
    oof_auc.append(auc)
    test_preds += model.predict(X_test) / N_SPLITS
    importance_df[f"fold{fold}"] = model.feature_importance(importance_type="gain")
    print(f"  Fold {fold}: AUC = {auc:.4f}  (best_iter={model.best_iteration})")

print(f"\nOOF AUC: {np.mean(oof_auc):.4f} ± {np.std(oof_auc):.4f}")

# ── 重要度の fold 平均 & 標準偏差を集計 ──────────────────────────────────────
fold_cols = [f"fold{i}" for i in range(1, N_SPLITS + 1)]
importance_df["importance_mean"] = importance_df[fold_cols].mean(axis=1)
importance_df["importance_std"]  = importance_df[fold_cols].std(axis=1)
importance_df = importance_df.sort_values("importance_mean", ascending=False).reset_index(drop=True)

print("\n特徴量重要度（上位15件）:")
print(importance_df[["feature", "importance_mean", "importance_std"]].head(15).to_string(index=False))

# ── 可視化（Plotly 横棒グラフ・上位20件） ────────────────────────────────────
top_n = importance_df.head(20).iloc[::-1]

fig = go.Figure(
    go.Bar(
        y=top_n["feature"],
        x=top_n["importance_mean"],
        error_x=dict(type="data", array=top_n["importance_std"], visible=True),
        orientation="h",
        marker_color="#3366cc",
        hovertemplate="<b>%{y}</b><br>gain mean: %{x:.1f}<extra></extra>",
    )
)
fig.update_layout(
    title="LightGBM 特徴量重要度（gain・4fold 平均 ± std）― 評価指標: AUC",
    xaxis_title="importance (gain)",
    height=600,
    paper_bgcolor="white",
    plot_bgcolor="white",
    margin=dict(l=220, r=60, t=60, b=40),
)
fig.show()

In [49]:
## サブミッション CSV 作成（上のセルで計算済みの test_preds をそのまま使用）

SAVE_PATH = PROCESSED_DIR / "submission.csv"

submission = sample_submission[["id"]].copy()
submission["PitNextLap"] = test_preds

submission.to_csv(SAVE_PATH, index=False)

print(f"submission.csv を保存しました")
print(f"shape: {submission.shape}")
print(submission.head(10).to_string(index=False))
print(f"\nPitNextLap スコアの分布:")
print(submission["PitNextLap"].describe())

submission.csv を保存しました
shape: (188165, 2)
    id  PitNextLap
439140    0.011171
439141    0.009697
439142    0.023041
439143    0.301394
439144    0.936520
439145    0.398705
439146    0.004246
439147    0.082244
439148    0.086794
439149    0.008709

PitNextLap スコアの分布:
count    188165.000000
mean          0.298602
std           0.351565
min           0.000344
25%           0.015585
50%           0.093518
75%           0.633216
max           0.991782
Name: PitNextLap, dtype: float64


In [7]:
## Train / Test マージ — 特徴量の型と値の範囲

features = [
    "Stint",
    "Year",
    "Driver",
    "Race",
    "TyreLife",
    "RaceProgress",
    "Compound"
]

train_part = train[features].assign(_dataset="train")
test_part = test[features].assign(_dataset="test")
merged = pd.concat([train_part, test_part], ignore_index=True)

print(f"merged shape: {merged.shape[0]:,} rows × {len(features)} cols")
print(f"  train: {len(train_part):,}  |  test: {len(test_part):,}\n")

# ── 全体サマリ（型・欠損・ユニーク数・値の範囲）────────────────────────────
summary_rows = []
for col in features:
    s = merged[col]
    row = {
        "feature": col,
        "dtype": str(s.dtype),
        "null_count": int(s.isna().sum()),
        "null_ratio": s.isna().mean(),
        "n_unique": s.nunique(dropna=True),
    }
    if pd.api.types.is_numeric_dtype(s):
        valid = s.dropna()
        row["min"] = valid.min()
        row["max"] = valid.max()
        row["mean"] = valid.mean()
        row["median"] = valid.median()
    else:
        row["min"] = row["max"] = row["mean"] = row["median"] = pd.NA
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print("=== 全体（train + test）===")
display(summary_df)

# ── train / test 別の値の範囲（数値列）────────────────────────────────────
print("\n=== train / test 別の min・max（数値列）===")
for col in features:
    if not pd.api.types.is_numeric_dtype(merged[col]):
        continue
    by_ds = merged.groupby("_dataset")[col].agg(["min", "max", "mean", "median"])
    print(f"\n[{col}]")
    display(by_ds)

# ── カテゴリ列：ユニーク数と出現上位 ───────────────────────────────────────
print("\n=== カテゴリ列（Driver, Race）===")
for col in ["Driver", "Race"]:
    print(f"\n[{col}]  n_unique (merged): {merged[col].nunique()}")
    for ds in ["train", "test"]:
        vc = merged.loc[merged["_dataset"] == ds, col].value_counts().head(5)
        print(f"  top5 in {ds}:")
        display(vc.to_frame("count"))


merged shape: 627,305 rows × 7 cols
  train: 439,140  |  test: 188,165

=== 全体（train + test）===


,feature,dtype,null_count,null_ratio,n_unique,min,max,mean,median
0,Stint,int64,0,0.0,8,1,8,1.78765,2.0
1,Year,int64,0,0.0,4,2022,2025,2023.525013,2024.0
2,Driver,object,0,0.0,887,<NA>,<NA>,<NA>,<NA>
3,Race,object,0,0.0,26,<NA>,<NA>,<NA>,<NA>
4,TyreLife,float64,0,0.0,78,1.0,77.0,14.158949,12.0
5,RaceProgress,float64,0,0.0,2097,0.012821,1.0,0.337371,0.269231
6,Compound,object,0,0.0,5,<NA>,<NA>,<NA>,<NA>



=== train / test 別の min・max（数値列）===

[Stint]


,min,max,mean,median
_dataset,,,,
test,1,8,1.784237,2.0
train,1,8,1.789113,2.0



[Year]


...

,count
Driver,
MAS,743
WEB,741
GLO,735
BUT,730
KUB,706



[Race]  n_unique (merged): 26
  top5 in train:


,count
Race,
Dutch Grand Prix,24462
Mexico City Grand Prix,23672
Pre-Season Testing,22492
Hungarian Grand Prix,22481
Monaco Grand Prix,21539


  top5 in test:


,count
Race,
Dutch Grand Prix,10340
Mexico City Grand Prix,10296
Hungarian Grand Prix,9721
Pre-Season Testing,9647
Monaco Grand Prix,9173
